
# ♟️ Beat Stockfish — Auto-play + Full Decision Logging

This Colab lets you either **play manually** or let a **Colab Bot auto-play** against Stockfish.

## New in this version

- **Manual mode** — you enter SAN/UCI moves.
- **Auto-play mode** — the Colab Bot plays your side automatically.
- **Auto-play speed** — control the delay between plies.
- **Separate bot strength** — make the Colab Bot strong while handicapping the opponent.
- **Start / pause auto-play** — safe control over long games.
- **Max plies per run** — prevents an accidental unbounded loop.
- **Full decision logging after every ply** — FEN, candidates, evaluations, chosen move, settings, timestamps, and results.
- **PGN snapshots after every move** — each game remains recoverable even if the runtime stops.
- **CSV + JSONL + PGN** logs.
- **Optional Google Drive persistence** — mount Drive before starting the game.
- **Download Logs ZIP** button.

### A good "Colab should win" configuration

- Colab Bot: **Skill 20**, think `0.20–0.50s`
- Opponent: **Skill 0–3**
- Win-friendly mode: **ON**
- Opponent blunder chance: **50–75%**
- Auto delay: `0.05–0.25s`

> Full-strength Stockfish vs full-strength Stockfish usually trends toward draws. To make the Colab Bot win more often, give the opponent a handicap rather than pretending unrestricted Stockfish is easy.


## v3 — Play Hint

v3 adds **▶ Play Hint** in Manual mode.

- **Hint**: analyze and show the best move without changing the board.
- **▶ Play Hint**: analyze, immediately play the best hint for your side, save the full decision, then let the opponent respond.

Play Hint decisions are tagged separately in the logs as:

- actor: `Human via Play Hint`
- selection reason: `play_hint_best_candidate`
- event: `play_hint_requested`

This keeps assisted moves distinguishable from moves you entered yourself.


In [ ]:

# Install engine + Python dependencies
!apt-get -qq update
!apt-get -qq install -y stockfish
!pip -q install python-chess ipywidgets pandas

print("✅ Installed Stockfish, python-chess, ipywidgets, and pandas.")


## Optional: save directly to Google Drive
Run the next cell **before** the main game cell if you want persistence across Colab runtimes.

In [ ]:

# OPTIONAL — mount Google Drive BEFORE running the game cell if you want logs
# to survive Colab runtime deletion.
#
# If you skip this cell, logs are saved under /content/chess_logs and can be
# downloaded with the "Download Logs ZIP" button.

from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted. The game will automatically save under MyDrive/ChessColabLogs.")


In [ ]:

import asyncio
import csv
import json
import os
import random
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import chess.svg
import ipywidgets as widgets
from IPython.display import display, SVG, clear_output

# ============================================================
# Engine setup
# ============================================================

candidate_paths = [
    shutil.which("stockfish"),
    "/usr/games/stockfish",
    "/usr/bin/stockfish",
]
STOCKFISH_PATH = next((p for p in candidate_paths if p and os.path.exists(p)), None)
if STOCKFISH_PATH is None:
    raise FileNotFoundError("Stockfish was not found. Re-run the install cell.")

engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)

# ============================================================
# Persistent session storage
# ============================================================

def utc_now():
    return datetime.now(timezone.utc).isoformat()

SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
DRIVE_BASE = Path("/content/drive/MyDrive")
if DRIVE_BASE.exists():
    LOG_ROOT = DRIVE_BASE / "ChessColabLogs"
    STORAGE_KIND = "Google Drive"
else:
    LOG_ROOT = Path("/content/chess_logs")
    STORAGE_KIND = "Colab local storage"

SESSION_DIR = LOG_ROOT / SESSION_ID
PGN_DIR = SESSION_DIR / "pgn"
SESSION_DIR.mkdir(parents=True, exist_ok=True)
PGN_DIR.mkdir(parents=True, exist_ok=True)

DECISIONS_JSONL = SESSION_DIR / "decisions.jsonl"
MOVES_CSV = SESSION_DIR / "moves.csv"
EVENTS_JSONL = SESSION_DIR / "events.jsonl"
SESSION_META = SESSION_DIR / "session_meta.json"
RESULTS_CSV = SESSION_DIR / "results.csv"

session_meta = {
    "session_id": SESSION_ID,
    "created_utc": utc_now(),
    "stockfish_path": STOCKFISH_PATH,
    "storage_kind": STORAGE_KIND,
    "session_dir": str(SESSION_DIR),
    "format_version": 2,
}
SESSION_META.write_text(json.dumps(session_meta, indent=2), encoding="utf-8")

CSV_FIELDS = [
    "session_id", "game_id", "decision_id", "timestamp_utc", "ply",
    "fullmove_number", "actor", "actor_color", "mode", "fen_before",
    "move_uci", "move_san", "chosen_rank", "chosen_score_cp", "mate_in",
    "selection_reason", "fen_after", "game_over", "result",
    "opponent_skill", "opponent_think_s", "opponent_blunder_pct",
    "colab_bot_skill", "colab_bot_think_s", "auto_delay_s",
]

if not MOVES_CSV.exists():
    with MOVES_CSV.open("w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

RESULT_FIELDS = ["session_id", "game_id", "ended_utc", "result", "termination", "plies", "mode"]
if not RESULTS_CSV.exists():
    with RESULTS_CSV.open("w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=RESULT_FIELDS).writeheader()

# ============================================================
# Game state
# ============================================================

board = chess.Board()
last_move = None
game_number = 0
game_id = None
decision_number = 0
current_game_finished_logged = False
auto_running = False
auto_task = None

# ============================================================
# Widgets
# ============================================================

mode_dropdown = widgets.Dropdown(
    options=[("Manual — I play", "manual"), ("Auto-play — Colab Bot plays", "auto")],
    value="manual",
    description="Mode:"
)

side_dropdown = widgets.Dropdown(
    options=[("White", chess.WHITE), ("Black", chess.BLACK)],
    value=chess.WHITE,
    description="Your/Bot side:"
)

opponent_skill_slider = widgets.IntSlider(value=0, min=0, max=20, step=1, description="Opp skill:")
opponent_think_slider = widgets.FloatSlider(
    value=0.10, min=0.02, max=3.0, step=0.02, readout_format=".2f", description="Opp think:"
)
friendly_checkbox = widgets.Checkbox(value=True, description="Win-friendly opponent")
blunder_slider = widgets.IntSlider(value=60, min=0, max=100, step=5, description="Opp blunder %:")

bot_skill_slider = widgets.IntSlider(value=20, min=0, max=20, step=1, description="Bot skill:")
bot_think_slider = widgets.FloatSlider(
    value=0.25, min=0.02, max=3.0, step=0.02, readout_format=".2f", description="Bot think:"
)

auto_delay_slider = widgets.FloatSlider(
    value=0.35,
    min=0.10,
    max=3.0,
    step=0.05,
    readout_format=".2f",
    description="Move delay:"
)
max_plies_slider = widgets.IntSlider(value=200, min=10, max=500, step=10, description="Max plies:")

move_box = widgets.Text(placeholder="e4, Nf3, O-O, e2e4 ...", description="Your move:")
play_button = widgets.Button(description="Play move", button_style="success")
hint_button = widgets.Button(description="Hint", button_style="info")
play_hint_button = widgets.Button(description="▶ Play Hint", button_style="primary")
undo_button = widgets.Button(description="Undo", button_style="warning")
reset_button = widgets.Button(description="New game")
start_auto_button = widgets.Button(description="▶ Start auto", button_style="success")
pause_auto_button = widgets.Button(description="⏸ Pause auto", button_style="warning")
download_button = widgets.Button(description="⬇ Download Logs ZIP", button_style="info")

board_output = widgets.Output()
status_output = widgets.Output()
log_output = widgets.Output()

autoplay_status = widgets.HTML(
    value="<b>Auto-play:</b> stopped"
)

# ============================================================
# Logging helpers
# ============================================================

def durable_flush(f):
    f.flush()
    try:
        os.fsync(f.fileno())
    except OSError:
        # Some mounted filesystems may not expose fsync cleanly.
        pass


def append_jsonl(path, payload):
    with Path(path).open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")
        durable_flush(f)


def log_event(event_type, **details):
    append_jsonl(EVENTS_JSONL, {
        "session_id": SESSION_ID,
        "game_id": game_id,
        "timestamp_utc": utc_now(),
        "event_type": event_type,
        "details": details,
    })


def score_payload(info, pov_color):
    raw = info.get("score")
    if raw is None:
        return {"score_cp": None, "mate_in": None}
    try:
        score = raw.pov(pov_color)
        return {
            "score_cp": score.score(mate_score=100000),
            "mate_in": score.mate(),
        }
    except Exception:
        return {"score_cp": None, "mate_in": None}


def settings_snapshot():
    return {
        "mode": mode_dropdown.value,
        "side": "white" if side_dropdown.value == chess.WHITE else "black",
        "opponent_skill": int(opponent_skill_slider.value),
        "opponent_think_s": float(opponent_think_slider.value),
        "opponent_win_friendly": bool(friendly_checkbox.value),
        "opponent_blunder_pct": int(blunder_slider.value),
        "colab_bot_skill": int(bot_skill_slider.value),
        "colab_bot_think_s": float(bot_think_slider.value),
        "auto_delay_s": float(auto_delay_slider.value),
        "max_plies_per_run": int(max_plies_slider.value),
    }


def actor_name_for_color(color):
    if color == side_dropdown.value:
        return "Human" if mode_dropdown.value == "manual" else "Colab Bot"
    return "Stockfish Opponent"


def build_game_pgn():
    game = chess.pgn.Game()
    game.headers["Event"] = "Beat Stockfish Colab"
    game.headers["Site"] = "Google Colab"
    game.headers["Date"] = datetime.now(timezone.utc).strftime("%Y.%m.%d")
    game.headers["Round"] = str(game_number)
    if side_dropdown.value == chess.WHITE:
        game.headers["White"] = actor_name_for_color(chess.WHITE)
        game.headers["Black"] = actor_name_for_color(chess.BLACK)
    else:
        game.headers["White"] = actor_name_for_color(chess.WHITE)
        game.headers["Black"] = actor_name_for_color(chess.BLACK)
    game.headers["GameID"] = str(game_id)
    game.headers["Mode"] = mode_dropdown.value
    game.headers["Result"] = board.result() if board.is_game_over() else "*"

    node = game
    temp = chess.Board()
    for mv in board.move_stack:
        node = node.add_variation(mv)
        temp.push(mv)
    return game


def save_pgn_snapshot():
    if game_id is None:
        return
    game = build_game_pgn()
    path = PGN_DIR / f"{game_id}.pgn"
    with path.open("w", encoding="utf-8") as f:
        print(game, file=f, end="\n\n")
        durable_flush(f)


def log_result_if_needed():
    global current_game_finished_logged
    if current_game_finished_logged or not board.is_game_over():
        return
    outcome = board.outcome()
    row = {
        "session_id": SESSION_ID,
        "game_id": game_id,
        "ended_utc": utc_now(),
        "result": board.result(),
        "termination": str(outcome.termination) if outcome else "unknown",
        "plies": len(board.move_stack),
        "mode": mode_dropdown.value,
    }
    with RESULTS_CSV.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        writer.writerow(row)
        durable_flush(f)
    current_game_finished_logged = True
    log_event("game_finished", **row)
    save_pgn_snapshot()


def record_decision(*, actor, fen_before, move, san, candidates, chosen_rank,
                    chosen_score_cp, mate_in, selection_reason, attempted_text=None):
    global decision_number
    decision_number += 1
    fen_after = board.fen()
    now = utc_now()
    s = settings_snapshot()

    detailed = {
        "session_id": SESSION_ID,
        "game_id": game_id,
        "decision_id": decision_number,
        "timestamp_utc": now,
        "ply": len(board.move_stack),
        "fullmove_number": board.fullmove_number,
        "actor": actor,
        "actor_color": "white" if not board.turn else "black",  # board.turn is next player after push
        "mode": mode_dropdown.value,
        "fen_before": fen_before,
        "attempted_text": attempted_text,
        "candidate_moves": candidates,
        "chosen": {
            "uci": move.uci(),
            "san": san,
            "rank": chosen_rank,
            "score_cp": chosen_score_cp,
            "mate_in": mate_in,
        },
        "selection_reason": selection_reason,
        "fen_after": fen_after,
        "game_over": board.is_game_over(),
        "result": board.result() if board.is_game_over() else "*",
        "settings": s,
    }
    append_jsonl(DECISIONS_JSONL, detailed)

    row = {
        "session_id": SESSION_ID,
        "game_id": game_id,
        "decision_id": decision_number,
        "timestamp_utc": now,
        "ply": len(board.move_stack),
        "fullmove_number": board.fullmove_number,
        "actor": actor,
        "actor_color": detailed["actor_color"],
        "mode": mode_dropdown.value,
        "fen_before": fen_before,
        "move_uci": move.uci(),
        "move_san": san,
        "chosen_rank": chosen_rank,
        "chosen_score_cp": chosen_score_cp,
        "mate_in": mate_in,
        "selection_reason": selection_reason,
        "fen_after": fen_after,
        "game_over": board.is_game_over(),
        "result": board.result() if board.is_game_over() else "*",
        "opponent_skill": s["opponent_skill"],
        "opponent_think_s": s["opponent_think_s"],
        "opponent_blunder_pct": s["opponent_blunder_pct"],
        "colab_bot_skill": s["colab_bot_skill"],
        "colab_bot_think_s": s["colab_bot_think_s"],
        "auto_delay_s": s["auto_delay_s"],
    }
    with MOVES_CSV.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writerow(row)
        durable_flush(f)

    save_pgn_snapshot()
    log_result_if_needed()

# ============================================================
# Game lifecycle
# ============================================================

def start_new_game(reason="new_game"):
    global board, last_move, game_number, game_id, decision_number, current_game_finished_logged
    board = chess.Board()
    last_move = None
    game_number += 1
    game_id = f"game_{game_number:04d}_{uuid.uuid4().hex[:6]}"
    decision_number = 0
    current_game_finished_logged = False
    log_event(reason, settings=settings_snapshot())
    save_pgn_snapshot()

# ============================================================
# UI rendering
# ============================================================

def render_board(message=None):
    with board_output:
        clear_output(wait=True)
        display(SVG(chess.svg.board(
            board=board,
            orientation=side_dropdown.value,
            lastmove=last_move,
            size=520,
        )))
        if board.is_game_over():
            print(f"\n🏁 Game over: {board.result()} — {board.outcome()}")
        elif message:
            print(f"\n{message}")
        else:
            print(f"\n{'White' if board.turn == chess.WHITE else 'Black'} to move")


def refresh_control_state():
    auto = mode_dropdown.value == "auto"
    move_box.disabled = auto
    play_button.disabled = auto
    hint_button.disabled = auto
    play_hint_button.disabled = auto
    start_auto_button.disabled = not auto or auto_running or board.is_game_over()
    pause_auto_button.disabled = not auto or not auto_running
    bot_skill_slider.disabled = not auto
    bot_think_slider.disabled = not auto
    auto_delay_slider.disabled = not auto
    max_plies_slider.disabled = not auto

# ============================================================
# Move / analysis logic
# ============================================================

def configure_skill(skill):
    try:
        engine.configure({"Skill Level": int(skill)})
    except Exception:
        pass


def analyse_candidates(skill, think_s, max_candidates=8):
    configure_skill(skill)
    legal_count = board.legal_moves.count()
    if legal_count == 0:
        return []
    k = min(max_candidates, legal_count)
    infos = engine.analyse(
        board,
        chess.engine.Limit(time=float(think_s)),
        multipv=max(1, k),
    )
    if isinstance(infos, dict):
        infos = [infos]

    out = []
    pov = board.turn
    for rank, info in enumerate(infos, start=1):
        pv = info.get("pv", [])
        if not pv:
            continue
        move = pv[0]
        if move not in board.legal_moves:
            continue
        sc = score_payload(info, pov)
        out.append({
            "rank": rank,
            "move": move,
            "uci": move.uci(),
            "san": board.san(move),
            "score_cp": sc["score_cp"],
            "mate_in": sc["mate_in"],
            "depth": info.get("depth"),
            "nodes": info.get("nodes"),
            "pv_uci": [m.uci() for m in pv[:12]],
        })
    return out


def push_engine_decision(actor, candidates, chosen, selection_reason):
    global last_move
    fen_before = board.fen()
    move = chosen["move"]
    san = board.san(move)
    board.push(move)
    last_move = move

    serializable_candidates = [
        {k: v for k, v in c.items() if k != "move"}
        for c in candidates
    ]
    record_decision(
        actor=actor,
        fen_before=fen_before,
        move=move,
        san=san,
        candidates=serializable_candidates,
        chosen_rank=chosen.get("rank"),
        chosen_score_cp=chosen.get("score_cp"),
        mate_in=chosen.get("mate_in"),
        selection_reason=selection_reason,
    )
    return san


def stockfish_opponent_turn():
    if board.is_game_over():
        return
    candidates = analyse_candidates(
        opponent_skill_slider.value,
        opponent_think_slider.value,
        max_candidates=8,
    )
    if not candidates:
        return

    if not friendly_checkbox.value or len(candidates) == 1:
        chosen = candidates[0]
        reason = "stockfish_best"
    else:
        p = blunder_slider.value / 100.0
        if random.random() > p:
            chosen = candidates[0]
            reason = "win_friendly_kept_best"
        else:
            start = max(1, len(candidates) // 2)
            weaker = candidates[start:] or candidates[1:] or candidates
            weights = list(range(1, len(weaker) + 1))
            chosen = random.choices(weaker, weights=weights, k=1)[0]
            reason = "win_friendly_weaker_multipv"

    san = push_engine_decision("Stockfish Opponent", candidates, chosen, reason)
    with status_output:
        clear_output(wait=True)
        print(f"🤖 Opponent played {san} ({chosen['uci']}) — candidate rank {chosen['rank']}")
    render_board()


def colab_bot_turn():
    if board.is_game_over():
        return
    candidates = analyse_candidates(
        bot_skill_slider.value,
        bot_think_slider.value,
        max_candidates=8,
    )
    if not candidates:
        return
    chosen = candidates[0]
    san = push_engine_decision("Colab Bot", candidates, chosen, "colab_bot_best")
    with status_output:
        clear_output(wait=True)
        print(f"🧠 Colab Bot played {san} ({chosen['uci']}) — score {chosen['score_cp']} cp")
    render_board()


def parse_user_move(text):
    text = text.strip()
    if not text:
        raise ValueError("Enter a move first.")
    try:
        return board.parse_san(text)
    except Exception:
        pass
    try:
        move = chess.Move.from_uci(text.lower())
        if move in board.legal_moves:
            return move
    except Exception:
        pass
    raise ValueError(f"'{text}' is not a legal SAN or UCI move in this position.")

# ============================================================
# Manual callbacks
# ============================================================

def on_play_clicked(_):
    global last_move
    if mode_dropdown.value != "manual":
        return
    if board.is_game_over():
        render_board("Game is over. Start a new game.")
        return
    if board.turn != side_dropdown.value:
        render_board("It is the opponent's turn.")
        return

    attempted = move_box.value
    try:
        move = parse_user_move(attempted)
        fen_before = board.fen()
        san = board.san(move)
        board.push(move)
        last_move = move
        move_box.value = ""
        record_decision(
            actor="Human",
            fen_before=fen_before,
            move=move,
            san=san,
            candidates=[],
            chosen_rank=None,
            chosen_score_cp=None,
            mate_in=None,
            selection_reason="manual_user_input",
            attempted_text=attempted,
        )
        with status_output:
            clear_output(wait=True)
            print(f"🧑 You played {san} ({move.uci()})")
        render_board()
        if not board.is_game_over():
            stockfish_opponent_turn()
    except Exception as exc:
        log_event("illegal_move_attempt", text=attempted, error=str(exc), fen=board.fen())
        with status_output:
            clear_output(wait=True)
            print(f"❌ {exc}")


def on_hint_clicked(_):
    if mode_dropdown.value != "manual" or board.is_game_over() or board.turn != side_dropdown.value:
        return
    candidates = analyse_candidates(bot_skill_slider.value, max(0.15, bot_think_slider.value), max_candidates=5)
    if candidates:
        best = candidates[0]
        log_event("hint_requested", fen=board.fen(), candidates=[{k:v for k,v in c.items() if k != "move"} for c in candidates])
        with status_output:
            clear_output(wait=True)
            print(f"💡 Hint: {best['san']} ({best['uci']}) — {best['score_cp']} cp")


def on_play_hint_clicked(_):
    """
    Analyse the user's current position with the Hint engine and immediately
    play its top candidate. The decision is stored through the same durable
    logging path as all other engine-generated moves.
    """
    if mode_dropdown.value != "manual":
        with status_output:
            clear_output(wait=True)
            print("▶ Play Hint is available in Manual mode.")
        return

    if board.is_game_over():
        render_board("Game is over. Start a new game.")
        return

    if board.turn != side_dropdown.value:
        render_board("It is the opponent's turn.")
        return

    fen_requested = board.fen()

    candidates = analyse_candidates(
        bot_skill_slider.value,
        max(0.15, bot_think_slider.value),
        max_candidates=8,
    )

    if not candidates:
        with status_output:
            clear_output(wait=True)
            print("No legal hint move is available.")
        return

    chosen = candidates[0]

    # Keep a dedicated event so assisted moves are easy to filter later.
    log_event(
        "play_hint_requested",
        fen=fen_requested,
        chosen_uci=chosen["uci"],
        chosen_san=chosen["san"],
        chosen_rank=chosen["rank"],
        chosen_score_cp=chosen["score_cp"],
        mate_in=chosen["mate_in"],
        candidates=[
            {k: v for k, v in c.items() if k != "move"}
            for c in candidates
        ],
    )

    san = push_engine_decision(
        actor="Human via Play Hint",
        candidates=candidates,
        chosen=chosen,
        selection_reason="play_hint_best_candidate",
    )

    move_box.value = ""

    with status_output:
        clear_output(wait=True)
        if chosen["mate_in"] is not None:
            evaluation = f"mate {chosen['mate_in']}"
        elif chosen["score_cp"] is not None:
            evaluation = f"{chosen['score_cp']} cp"
        else:
            evaluation = "score unavailable"

        print(
            f"▶💡 Played hint: {san} ({chosen['uci']}) — "
            f"candidate #{chosen['rank']}, {evaluation}"
        )

    render_board()

    # In Manual mode, immediately continue with the opponent reply.
    if not board.is_game_over():
        stockfish_opponent_turn()


def on_undo_clicked(_):
    global last_move
    if auto_running:
        return
    pops = min(2, len(board.move_stack))
    removed = []
    for _ in range(pops):
        removed.append(board.pop().uci())
    last_move = board.peek() if board.move_stack else None
    log_event("undo", removed_moves=removed, resulting_fen=board.fen())
    save_pgn_snapshot()
    render_board("↩️ Took back the last move pair.")


def on_reset_clicked(_):
    global auto_running
    auto_running = False
    if not board.is_game_over() and board.move_stack:
        log_event("game_abandoned_for_new_game", plies=len(board.move_stack), fen=board.fen())
    start_new_game("new_game")
    move_box.value = ""
    refresh_control_state()
    render_board("New game ready.")
    if mode_dropdown.value == "manual" and board.turn != side_dropdown.value:
        stockfish_opponent_turn()

# ============================================================
# Auto-play
# ============================================================


async def autoplay_loop():
    """
    Auto-play one ply at a time and explicitly yield back to the Jupyter/
    browser event loop after every board render.

    The important part for v4 is the two-stage yield:
      1) await asyncio.sleep(0) immediately after the move/render so the
         updated SVG board can be painted by Colab.
      2) await asyncio.sleep(move_delay) so that painted position stays
         visible before the next engine decision begins.

    This prevents fast auto-play from looking like the board jumped directly
    to a later/final position.
    """
    global auto_running

    start_ply = len(board.move_stack)
    target_ply = start_ply + int(max_plies_slider.value)

    log_event(
        "autoplay_started",
        start_ply=start_ply,
        target_ply=target_ply,
        visual_move_delay=float(auto_delay_slider.value),
    )

    try:
        # Paint the initial auto-play position before the first engine call.
        render_board("Auto-play starting…")
        autoplay_status.value = (
            f"<b>Auto-play:</b> running — ply {len(board.move_stack)}"
        )
        await asyncio.sleep(0.05)

        while (
            auto_running
            and not board.is_game_over()
            and len(board.move_stack) < target_ply
        ):
            mover = "Colab Bot" if board.turn == side_dropdown.value else "Stockfish Opponent"

            autoplay_status.value = (
                f"<b>Auto-play:</b> {mover} thinking — "
                f"ply {len(board.move_stack) + 1}"
            )

            # Yield first so the "thinking" state and current board get painted.
            await asyncio.sleep(0)

            ply_before = len(board.move_stack)

            if board.turn == side_dropdown.value:
                colab_bot_turn()
            else:
                stockfish_opponent_turn()

            ply_after = len(board.move_stack)

            # If an engine failed to produce a legal move, stop cleanly rather
            # than spinning forever on the same position.
            if ply_after == ply_before:
                log_event(
                    "autoplay_no_move",
                    ply=ply_before,
                    fen=board.fen(),
                    mover=mover,
                )
                with status_output:
                    clear_output(wait=True)
                    print("⏹ Auto-play stopped because no move was produced.")
                break

            # Both turn functions call render_board(). Force control back to
            # Colab so that SVG update is sent to the browser NOW.
            autoplay_status.value = (
                f"<b>Auto-play:</b> displayed ply {ply_after} — "
                f"{'White' if board.turn == chess.WHITE else 'Black'} to move"
            )
            refresh_control_state()
            await asyncio.sleep(0)

            if board.is_game_over():
                break

            # Keep this exact board position visible for the user's selected
            # speed before calculating the next move.
            visible_delay = max(0.10, float(auto_delay_slider.value))
            await asyncio.sleep(visible_delay)

    except asyncio.CancelledError:
        log_event(
            "autoplay_task_cancelled",
            ply=len(board.move_stack),
            fen=board.fen(),
        )
        raise

    except Exception as exc:
        log_event(
            "autoplay_error",
            error=repr(exc),
            ply=len(board.move_stack),
            fen=board.fen(),
        )
        with status_output:
            clear_output(wait=True)
            print(f"❌ Auto-play stopped: {exc}")

    finally:
        auto_running = False
        refresh_control_state()
        log_result_if_needed()

        if board.is_game_over():
            autoplay_status.value = (
                f"<b>Auto-play:</b> finished — {board.result()}"
            )
            with status_output:
                clear_output(wait=True)
                print(f"🏁 Auto-play finished: {board.result()}")

        elif len(board.move_stack) >= target_ply:
            autoplay_status.value = (
                f"<b>Auto-play:</b> paused at run limit — "
                f"ply {len(board.move_stack)}"
            )
            with status_output:
                clear_output(wait=True)
                print(
                    f"⏹ Auto-play reached the {max_plies_slider.value}-ply "
                    "run limit. Press Start auto to continue."
                )

        else:
            autoplay_status.value = (
                f"<b>Auto-play:</b> stopped — ply {len(board.move_stack)}"
            )

        # Final explicit paint/yield for the final position.
        render_board()
        await asyncio.sleep(0)

def on_start_auto(_):
    global auto_running, auto_task
    if mode_dropdown.value != "auto" or auto_running or board.is_game_over():
        return
    auto_running = True
    autoplay_status.value = (
        f"<b>Auto-play:</b> starting — ply {len(board.move_stack)}"
    )
    refresh_control_state()
    auto_task = asyncio.create_task(autoplay_loop())


def on_pause_auto(_):
    global auto_running
    auto_running = False
    autoplay_status.value = (
        f"<b>Auto-play:</b> pause requested — ply {len(board.move_stack)}"
    )
    log_event(
        "autoplay_pause_requested",
        ply=len(board.move_stack),
        fen=board.fen(),
    )
    refresh_control_state()

# ============================================================
# Download + setting changes
# ============================================================

def on_download_clicked(_):
    save_pgn_snapshot()
    zip_base = str(SESSION_DIR)
    zip_path = shutil.make_archive(zip_base, "zip", root_dir=SESSION_DIR)
    log_event("logs_zipped", zip_path=zip_path)
    with log_output:
        clear_output(wait=True)
        print(f"📦 Log archive created: {zip_path}")
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            print("Download API unavailable; use the path above.")


def on_setting_change(change):
    if change.get("name") == "value":
        owner = change.get("owner")
        desc = getattr(owner, "description", "setting")
        log_event("setting_changed", setting=desc, old=change.get("old"), new=change.get("new"))
        refresh_control_state()


def on_mode_or_side_change(change):
    global auto_running
    if change.get("name") != "value":
        return
    auto_running = False
    setting = getattr(change.get("owner"), "description", "")
    log_event("mode_or_side_changed", setting=setting, old=change.get("old"), new=change.get("new"))

    # A mode/side change starts a clean game so turn ownership cannot become
    # inconsistent with the board already in progress. The previous PGN/logs
    # are already preserved under their original game_id.
    if game_id is not None:
        if board.move_stack and not board.is_game_over():
            log_event("game_abandoned_for_mode_or_side_change", plies=len(board.move_stack), fen=board.fen())
        start_new_game("mode_or_side_new_game")

    autoplay_status.value = "<b>Auto-play:</b> stopped"
    refresh_control_state()
    render_board("Mode/side changed — started a new game.")
    if mode_dropdown.value == "manual" and board.turn != side_dropdown.value:
        stockfish_opponent_turn()

# Wire callbacks
play_button.on_click(on_play_clicked)
hint_button.on_click(on_hint_clicked)
play_hint_button.on_click(on_play_hint_clicked)
undo_button.on_click(on_undo_clicked)
reset_button.on_click(on_reset_clicked)
start_auto_button.on_click(on_start_auto)
pause_auto_button.on_click(on_pause_auto)
download_button.on_click(on_download_clicked)

mode_dropdown.observe(on_mode_or_side_change, names="value")
side_dropdown.observe(on_mode_or_side_change, names="value")
for w in [opponent_skill_slider, opponent_think_slider, friendly_checkbox, blunder_slider,
          bot_skill_slider, bot_think_slider, auto_delay_slider, max_plies_slider]:
    w.observe(on_setting_change, names="value")

# ============================================================
# Layout
# ============================================================

storage_html = widgets.HTML(
    value=(
        f"<b>Storage:</b> {STORAGE_KIND}<br>"
        f"<code>{SESSION_DIR}</code><br>"
        "Each ply is flushed to CSV/JSONL and the current PGN is rewritten immediately."
    )
)

controls = widgets.VBox([
    widgets.HTML("<h3>🎮 Game</h3>"),
    mode_dropdown,
    side_dropdown,
    widgets.HTML("<h4>🤖 Stockfish opponent</h4>"),
    opponent_skill_slider,
    opponent_think_slider,
    friendly_checkbox,
    blunder_slider,
    widgets.HTML("<h4>🧠 Colab Bot / Hint engine</h4>"),
    bot_skill_slider,
    bot_think_slider,
    widgets.HTML("<h4>⚡ Auto-play</h4>"),
    auto_delay_slider,
    max_plies_slider,
    widgets.HBox([start_auto_button, pause_auto_button]),
    autoplay_status,
    widgets.HTML(
        "<small>v4 renders and yields after every ply so the board visibly "
        "updates before the next move.</small>"
    ),
    widgets.HTML("<h4>✍️ Manual play</h4>"),
    move_box,
    widgets.HBox([play_button, hint_button, play_hint_button]),
    widgets.HBox([undo_button, reset_button]),
    widgets.HTML("<h4>💾 Logs</h4>"),
    download_button,
    storage_html,
])

display(widgets.HBox([controls, widgets.VBox([board_output, status_output, log_output])]))

start_new_game("session_first_game")
refresh_control_state()
render_board("Ready. Manual mode is active.")

print(f"✅ Stockfish: {STOCKFISH_PATH}")
print(f"✅ Session ID: {SESSION_ID}")
print(f"✅ Logs: {SESSION_DIR}")



## What is saved?

Every move is saved immediately in two forms:

- `moves.csv` — compact, analysis-friendly summary per ply.
- `decisions.jsonl` — complete decision record including all MultiPV candidates returned for that decision, their scores, PVs, chosen rank, settings, FEN before/after, and selection reason.

Additional files:

- `events.jsonl` — resets, hints, illegal move attempts, setting changes, auto-play start/pause/errors, etc.
- `results.csv` — completed game results.
- `pgn/<game_id>.pgn` — the current PGN snapshot, rewritten **after every move**.
- `session_meta.json` — session metadata and storage location.

If Drive is mounted first, the session is written under:

`MyDrive/ChessColabLogs/<session_id>/`

Otherwise it is written under `/content/chess_logs/<session_id>/`; use **Download Logs ZIP** before the runtime disappears.


In [ ]:

# Optional: inspect the saved move log as a DataFrame.
import pandas as pd
from IPython.display import display

if MOVES_CSV.exists():
    df = pd.read_csv(MOVES_CSV)
    display(df.tail(50))
else:
    print("No moves logged yet.")


In [ ]:

# Run this when you are done. It saves the latest PGN snapshot and shuts down Stockfish.
try:
    auto_running = False
    save_pgn_snapshot()
    log_event("session_shutdown_requested", final_fen=board.fen(), plies=len(board.move_stack))
except Exception:
    pass

try:
    engine.quit()
    print("✅ Stockfish shut down.")
except Exception:
    print("Stockfish was already stopped.")

print(f"Logs remain at: {SESSION_DIR}")
